# SMS Spam Modeling Pipeline

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = SparkSession.builder.appName('sms-spam-modeling').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

clean_df = spark.read.parquet('/content/artifacts/clean_sms.parquet')
print('Rows:', clean_df.count())
print('Columns:', clean_df.columns)
clean_df.show(5, truncate=False)

Rows: 5572
Columns: ['label', 'label_text', 'message', 'msg_len_chars', 'msg_len_words']
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|label|label_text|message                                                                                                                                                    |msg_len_chars|msg_len_words|
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|0    |ham       |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                            |111          |20           |
|0    |ham       |Ok lar... Joking wif u oni...                                                    

In [3]:
train_df, test_df = clean_df.select('label', 'message').randomSplit([0.8, 0.2], seed=42)
print('Train rows:', train_df.count())
print('Test rows:', test_df.count())

tokenizer = Tokenizer(inputCol='message', outputCol='tokens')
stopwords = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_features', numFeatures=2**16)
idf = IDF(inputCol='raw_features', outputCol='features')
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=30)

pipeline = Pipeline(stages=[tokenizer, stopwords, tf, idf, lr])
print('Pipeline ready')

Train rows: 4501
Test rows: 1071
Pipeline ready


In [4]:
model = pipeline.fit(train_df)
pred_df = model.transform(test_df)

pred_df.select('label', 'prediction', 'probability', 'message').show(10, truncate=False)

+-----+----------+-------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                |message                                                                                                                                                                                                                                                                                                                                                                                         |
+-----+----------+-------------------------------------------+------------------------------------------

In [5]:
tp = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 1)).count()
tn = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 0)).count()
fp = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 1)).count()
fn = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 0)).count()

total = tp + tn + fp + fn
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')

TP=130, TN=914, FP=9, FN=18
Accuracy : 0.9748
Precision: 0.9353
Recall   : 0.8784
F1-score : 0.9059


In [6]:
from pyspark.ml import PipelineModel

model_path = '/content/artifacts/models/sms_spam_pipeline'
model.write().overwrite().save(model_path)
print('Saved model to:', model_path)

loaded_model = PipelineModel.load(model_path)

new_messages = spark.createDataFrame(
    [
        ('Congratulations! You won a free ticket. Reply WIN now!',),
        ('Can we meet tomorrow at 10 to finish the report?',),
        ('URGENT: claim your reward by calling this number now',),
    ],
    ['message']
 )

new_pred = loaded_model.transform(new_messages)
new_pred.select('message', 'prediction', 'probability').show(truncate=False)

Saved model to: /content/artifacts/models/sms_spam_pipeline
+------------------------------------------------------+----------+------------------------------------------+
|message                                               |prediction|probability                               |
+------------------------------------------------------+----------+------------------------------------------+
|Congratulations! You won a free ticket. Reply WIN now!|1.0       |[1.123837476691011E-6,0.9999988761625233] |
|Can we meet tomorrow at 10 to finish the report?      |0.0       |[0.9999999992745252,7.254747913520987E-10]|
|URGENT: claim your reward by calling this number now  |0.0       |[0.9892299285182212,0.010770071481778776] |
+------------------------------------------------------+----------+------------------------------------------+



In [7]:
errors = pred_df.filter(F.col('label') != F.col('prediction'))
print('Misclassified rows:', errors.count())
errors.select('label', 'prediction', 'probability', 'message').show(20, truncate=False)

Misclassified rows: 27
+-----+----------+-------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                |message                                                                                                                                                                                                                                                                                                                           |
+-----+----------+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
import pandas as pd
from pathlib import Path

metrics_path = Path('/content/artifacts/model_metrics.csv')
metrics_table = pd.DataFrame(
    [
        {'metric': 'accuracy', 'value': accuracy},
        {'metric': 'precision', 'value': precision},
        {'metric': 'recall', 'value': recall},
        {'metric': 'f1_score', 'value': f1},
    ]
)
metrics_table.to_csv(metrics_path, index=False)
print('Saved metrics table to:', metrics_path)
metrics_table

Saved metrics table to: /content/artifacts/model_metrics.csv


,metric,value
0,accuracy,0.974790
1,precision,0.935252
2,recall,0.878378
3,f1_score,0.905923


In [9]:
base_df = clean_df.select('label', 'message')
train_df_v, val_df_v, test_df_v = base_df.randomSplit([0.7, 0.15, 0.15], seed=42)

print('Train rows:', train_df_v.count())
print('Validation rows:', val_df_v.count())
print('Test rows:', test_df_v.count())

def compute_binary_metrics(predictions):
    tp = predictions.filter((F.col('label') == 1) & (F.col('prediction') == 1)).count()
    tn = predictions.filter((F.col('label') == 0) & (F.col('prediction') == 0)).count()
    fp = predictions.filter((F.col('label') == 0) & (F.col('prediction') == 1)).count()
    fn = predictions.filter((F.col('label') == 1) & (F.col('prediction') == 0)).count()

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }

def build_pipeline(classifier):
    tokenizer = Tokenizer(inputCol='message', outputCol='tokens')
    stopwords = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
    tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_features', numFeatures=2**16)
    idf = IDF(inputCol='raw_features', outputCol='features')
    return Pipeline(stages=[tokenizer, stopwords, tf, idf, classifier])

Train rows: 3979
Validation rows: 787
Test rows: 806


In [10]:
lr_grid = [
    {'maxIter': 20, 'regParam': 0.0, 'elasticNetParam': 0.0},
    {'maxIter': 30, 'regParam': 0.05, 'elasticNetParam': 0.0},
    {'maxIter': 40, 'regParam': 0.1, 'elasticNetParam': 0.0},
    {'maxIter': 40, 'regParam': 0.05, 'elasticNetParam': 0.5},
    {'maxIter': 60, 'regParam': 0.1, 'elasticNetParam': 0.5},
]

tuning_rows = []
best_lr = None
best_lr_params = None
best_val_f1 = -1.0

for params in lr_grid:
    lr_candidate = LogisticRegression(featuresCol='features', labelCol='label', **params)
    candidate_pipeline = build_pipeline(lr_candidate)
    candidate_model = candidate_pipeline.fit(train_df_v)
    val_pred = candidate_model.transform(val_df_v)
    val_metrics = compute_binary_metrics(val_pred)

    tuning_rows.append({
        'model': 'LogisticRegression',
        **params,
        **val_metrics,
    })

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_lr = candidate_model
        best_lr_params = params

tuning_table = pd.DataFrame(tuning_rows).sort_values('f1', ascending=False)
tuning_table

,model,maxIter,regParam,elasticNetParam,accuracy,precision,recall,f1,tp,tn,fp,fn
0,LogisticRegression,20,0.00,0.0,0.980940,0.967391,0.881188,0.922280,89,683,3,12
1,LogisticRegression,30,0.05,0.0,0.954257,1.000000,0.643564,0.783133,65,686,0,36
2,LogisticRegression,40,0.10,0.0,0.952986,1.000000,0.633663,0.775758,64,686,0,37
3,LogisticRegression,40,0.05,0.5,0.932656,0.961538,0.495050,0.653595,50,684,2,51
4,LogisticRegression,60,0.10,0.5,0.897078,1.000000,0.198020,0.330579,20,686,0,81


In [11]:
print('Best LR params:', best_lr_params)

best_lr_test_pred = best_lr.transform(test_df_v)
best_lr_test_metrics = compute_binary_metrics(best_lr_test_pred)
best_lr_test_metrics

Best LR params: {'maxIter': 20, 'regParam': 0.0, 'elasticNetParam': 0.0}


{'accuracy': 0.9714640198511166,
 'precision': 0.9009009009009009,
 'recall': 0.8928571428571429,
 'f1': 0.8968609865470852,
 'tp': 100,
 'tn': 683,
 'fp': 11,
 'fn': 12}

In [12]:
from pyspark.ml.classification import LinearSVC, NaiveBayes

comparison_rows = [
    {'model': 'LogisticRegression_tuned', **best_lr_test_metrics}
 ]

lsvc = LinearSVC(featuresCol='features', labelCol='label', maxIter=40, regParam=0.1)
lsvc_model = build_pipeline(lsvc).fit(train_df_v)
lsvc_test_pred = lsvc_model.transform(test_df_v)
lsvc_metrics = compute_binary_metrics(lsvc_test_pred)
comparison_rows.append({'model': 'LinearSVC', **lsvc_metrics})

nb = NaiveBayes(featuresCol='features', labelCol='label', modelType='multinomial', smoothing=1.0)
nb_model = build_pipeline(nb).fit(train_df_v)
nb_test_pred = nb_model.transform(test_df_v)
nb_metrics = compute_binary_metrics(nb_test_pred)
comparison_rows.append({'model': 'NaiveBayes', **nb_metrics})

comparison_table = pd.DataFrame(comparison_rows).sort_values('f1', ascending=False)
comparison_table

,model,accuracy,precision,recall,f1,tp,tn,fp,fn
1,LinearSVC,0.978908,0.952381,0.892857,0.921659,100,689,5,12
0,LogisticRegression_tuned,0.971464,0.900901,0.892857,0.896861,100,683,11,12
2,NaiveBayes,0.955335,0.783582,0.937500,0.853659,105,665,29,7


In [13]:
comparison_path = Path('/content/artifacts/model_comparison.csv')
comparison_table.to_csv(comparison_path, index=False)
print('Saved model comparison to:', comparison_path)

Saved model comparison to: /content/artifacts/model_comparison.csv
